In [1]:
%load_ext autoreload
%autoreload 2

import logging

import pandas as pd

from checkerduck import log as logsetup
from checkerduck.db.db import init_database
from checkerduck.rules import debug
from checkerduck.rules.seo_rule import *          # re-run this cell if you ADD a rule class

# WARNING keeps the table readable; DEBUG to watch individual rules and their SQL.
logsetup.setup(logging.WARNING)

# Let the frontend render the full frame; it scrolls the output area natively,
# which is what makes trackpad scrolling work.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", None)
pd.set_option("display.precision", 2)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Input: domains from CSV



In [22]:
ROOT = debug.project_root()          # repo root, however the kernel was started

CSV_PATH   = ROOT / "data" / "candidate_domains.csv"
DB_PATH    = ROOT / "data" / "checkerduck.db"
NOTES_PATH = ROOT / "notebooks" / "rule_comments.json"
TARGET_ID  = "x"
LIMIT      = 200          # the CSV holds ~785 rows; extraction covered the first 200

init_database(db_path=str(DB_PATH), alembic_ini_path=str(ROOT / "alembic.ini") )

DOMAINS = pd.read_csv(CSV_PATH)["url"].dropna().unique().tolist()[:LIMIT]

# Your notes per domain. A plain JSON file, so they survive kernel restarts.
COMMENTS = debug.Comments(NOTES_PATH)

print(f"root     {ROOT}")
print(f"domains  {len(DOMAINS)} from {CSV_PATH.relative_to(ROOT)}, e.g. {DOMAINS[:3]}")
print(f"notes    {COMMENTS}")

DOMAINS

root     /Users/antonpolyakov/dev/checkerduck
domains  200 from data/candidate_domains.csv, e.g. ['amourvert.com', 'arabamerica.com', 'arfadia.com']
notes    Comments('/Users/antonpolyakov/dev/checkerduck/notebooks/rule_comments.json', 3 notes)


['amourvert.com',
 'arabamerica.com',
 'arfadia.com',
 'authorea.com',
 'bd24live.com',
 'brewminate.com',
 'builtinseattle.com',
 'caclubindia.com',
 'clickdo.co.uk',
 'cssauthor.com',
 'culturecustodian.com',
 'cupidbaba.com',
 'd24am.com',
 'dailyexcelsior.com',
 'datafilehost.com',
 'diigo.com',
 'es.kompass.com',
 'extendsclass.com',
 'galiciapress.es',
 'gameluster.com',
 'gamespace.com',
 'growthhackers.com',
 'hacker9.com',
 'hive.com',
 'indiadeets.com',
 'indianagazette.com',
 'kanbanzone.com',
 'leavedates.com',
 'mspoweruser.com',
 'musculoskeletalkey.com',
 'picmaker.com',
 'rakeback.com',
 'rapidhomedirect.com',
 'rockawave.com',
 'stackademic.com',
 'taggbox.com',
 'tanks-encyclopedia.com',
 'tennessean.com',
 'the-art-world.com',
 'thesoundstour.com',
 'thinkofgames.com',
 'timesargus.com',
 'topescort.com',
 'veganovtrichy.com',
 'visitsouthdevon.co.uk',
 'walterfootball.com',
 'wealthylike.com',
 'xoxoday.com',
 'asiaone.com',
 'azbigmedia.com',
 'betgenuine.com',
 'b

## Coverage: what the rules have to work with


In [3]:
display(debug.coverage(TARGET_ID))

missing = debug.missing_domains(TARGET_ID, DOMAINS)
print(f"{len(missing)} of {len(DOMAINS)} input domains were never extracted"
      + (f", e.g. {missing[:5]}" if missing else ""))

,rows,domains
Table,,
batch_analysis,200,200
ahrefs_org_traffic_country,976,200
domain_categories,645,198
ahrefs_metrics_history,0,0
ahrefs_top_pages,0,0
ahrefs_backlinks,0,0
anchors_forbidden_words,0,0
ahrefs_organic_keywords,0,0


0 of 200 input domains were never extracted


## The rule set


## Review

Pick a domain, type a note. Saves to `rule_comments.json` on blur or Enter (one file write
per edit, not per keystroke), and re-reads the file on every dropdown change so notes edited
elsewhere show up. The note then appears in the `comment` column of the table below.

The dropdown is in CSV order. After a first run, `debug.review(COMMENTS,
run.table(ascending=True).index)` orders it worst-domain-first instead.


In [21]:

debug.review(COMMENTS, DOMAINS)

## Evaluate

In [23]:
# DOMAINS = ["theglobeandmail.com"]
RULES = [
    DomainRatingRule(),
    OrganicTrafficRule(),
    HistoricalOrganicTrafficRule(),          # weight=0.8, min_avg_traffic=50
    GeographyRule(),
    DomainsInOutLinksRatioRule(),
    SingleTopPageTrafficRule(),
    SpamWordsAnchorsRule(),                  # weight=1.0, spam_threshold=0.1
    ForbiddenWordsAnchorRule(),
    ForbiddenWordsOrganicKeywordsRule(),
    SpamWordsOrganicKeywordsRule(),
    DomainCategoryRule("Technology"),      # embedding rule: workers drops to 1 automatically
]
before = globals().get("run")        # kept so the Diff cell can compare

run = debug.evaluate(TARGET_ID, RULES, domains=DOMAINS)

debug.show(run.table(comments=COMMENTS))    # hide_constant=True also drops uninformative rules

16:01:53 WARNING forcing workers=1: ['DomainCategory'] hold a torch model, and concurrent inference aborts the process
domain amourvert.com, category ['Shopping & Auctions', 'Society & Lifestyle', 'Society & Lifestyle: Clothing', 'Shopping & Auctions: Ecommerce']
domain arabamerica.com, category ['Entertainment: News & Media', 'Entertainment', 'Entertainment']
domain arfadia.com, category ['Technology', 'Entertainment: Video Streaming', 'Technology', 'Business & Economy', 'Entertainment', 'Business & Economy: Business']
domain authorea.com, category ['Education: Science', 'Education', 'Education']
domain bd24live.com, category ['Entertainment: News & Media', 'Entertainment']
domain brewminate.com, category ['Entertainment: News & Media', 'Entertainment']
domain builtinseattle.com, category ['Business & Economy', 'Business & Economy: Business']
domain caclubindia.com, category ['Internet Communication', 'Business & Economy', 'Business & Economy: Economy & Finance', 'Internet Communicati

/Users/antonpolyakov/dev/checkerduck/checkerduck/rules/seo_rule.py:151: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  slope, intercept, r_value, p_value, std_err = linregress(x, traffic)


domain clickdo.co.uk, category ['Technology: Information Technology', 'Technology', 'Business & Economy', 'Business & Economy: Business']
domain cssauthor.com, category ['Society & Lifestyle: Hobbies & Interests', 'Technology: Information Technology', 'Society & Lifestyle', 'Technology']
domain culturecustodian.com, category ['Entertainment: Music', 'Entertainment: Video Streaming', 'Entertainment', 'Entertainment']
domain cupidbaba.com, category ['Adult Themes: Pornography', 'CIPA: CIPA Filter', 'Adult Themes', 'Shopping & Auctions', 'CIPA', 'Shopping & Auctions: Ecommerce']
domain d24am.com, category ['Entertainment: News & Media', 'Entertainment']
domain dailyexcelsior.com, category ['Entertainment: News & Media', 'Entertainment']
domain datafilehost.com, category ['Technology', 'Technology: File Sharing']
domain diigo.com, category ['Technology', 'Technology', 'Business & Economy', 'Business & Economy: Business']
domain es.kompass.com, category ['Technology: Search Engines', 'Techn

,Domain,comment,overall,weighted,critical,errors,DomainRating,OrganicTraffic,HistoricalOrganicTraffic,Geography,DomainsInOutLinksRatio,SingleTopPageTraffic,SpamWordsAnchors,ForbiddenWordsAnchor,ForbiddenWordsOrganicKeywords,SpamWordsOrganicKeywords,DomainCategory
0,theglobeandmail.com,,0.90,0.90,0,2,0.90,1.00,1.0,NaN,0.84,NaN,1.0,1.0,1.0,1.0,0.37
1,builtinnyc.com,,0.90,0.89,0,2,0.73,1.00,1.0,NaN,1.00,NaN,1.0,1.0,1.0,1.0,0.36
2,gametyrant.com,,0.90,0.89,0,2,0.62,1.00,1.0,NaN,1.00,NaN,1.0,1.0,1.0,1.0,0.44
3,thegww.com,,0.89,0.88,0,2,0.63,1.00,1.0,NaN,1.00,NaN,1.0,1.0,1.0,1.0,0.37
4,fightful.com,,0.88,0.88,0,2,0.63,1.00,1.0,NaN,0.88,NaN,1.0,1.0,1.0,1.0,0.40
5,feedinco.com,,0.87,0.86,0,2,0.52,1.00,1.0,NaN,1.00,NaN,1.0,1.0,1.0,1.0,0.31
6,topescort.com,,0.87,0.86,0,2,0.61,1.00,1.0,NaN,1.00,NaN,1.0,1.0,1.0,1.0,0.20
7,strikingly.com,,0.87,0.91,0,2,0.90,1.00,1.0,NaN,0.03,NaN,1.0,1.0,1.0,1.0,0.86
8,goonlinetools.com,,0.86,0.90,0,2,0.56,1.00,1.0,NaN,0.19,NaN,1.0,1.0,1.0,1.0,1.00
9,greenbuildingadvisor.com,,0.86,0.85,0,2,0.74,0.60,1.0,NaN,1.00,NaN,1.0,1.0,1.0,1.0,0.38


## Bulk comment editing

The Review control above covers one domain at a time. For many at once:


In [6]:
# several at once ('' clears a note)
COMMENTS.update({
    "eerone.com": "thin content",
})

# walk the worst domains on stdin; Enter skips, '-' clears, interrupt to stop
# COMMENTS.prompt(run.table(ascending=True).index[:10])

# round-trip through a spreadsheet
# COMMENTS.to_csv('notes.csv');  COMMENTS.read_csv('notes.csv')

COMMENTS

Comments('/Users/antonpolyakov/dev/checkerduck/notebooks/rule_comments.json', 3 notes)

## Drill down

Everything one rule set said about one domain, plus every rule that crashed.

In [12]:
worst = "theglobeandmail.com" #run.table(ascending=True).index[0]      # table() is best-first now
print(worst)
run.for_domain(worst)

theglobeandmail.com


,score,critical,details,error,name,weight,area
Rule,,,,,,,
DomainRating,0.90,False,,,Domain Rating,1.0,authority
OrganicTraffic,1.00,False,,,Organic Traffic,1.0,traffic
HistoricalOrganicTraffic,1.00,False,,,Historical Organic Traffic,0.8,traffic
Geography,NaN,False,,"Traceback (most recent call last):\n File ""/U...",Geography,1.0,relevance
DomainsInOutLinksRatio,0.84,False,,,Domains In/Out Links Ratio,0.5,authority
SingleTopPageTraffic,NaN,False,,"Traceback (most recent call last):\n File ""/U...",Single Top Page Traffic,1.0,relevance
SpamWordsAnchors,1.00,False,,,Spam Words in Anchors,1.0,safety
ForbiddenWordsAnchor,1.00,False,,,Spam Words in Anchors,1.0,safety
ForbiddenWordsOrganicKeywords,1.00,False,,,Forbidden Words in Organic Keywords,1.0,safety


In [8]:
fails = run.failures()
print(f"{len(fails)} of {run.scores.size} (domain, rule) evaluations raised")
if not fails.empty:
    print(fails.groupby("rule").size().to_string())
    print("\n" + fails.iloc[0]["error"])

403 of 2200 (domain, rule) evaluations raised
rule
DomainCategory            3
Geography               200
SingleTopPageTraffic    200

Traceback (most recent call last):
  File "/Users/antonpolyakov/dev/checkerduck/checkerduck/rules/debug.py", line 294, in one
    ev = rule.eval(ctx)
  File "/Users/antonpolyakov/dev/checkerduck/checkerduck/rules/seo_rule.py", line 170, in eval
    def get_domain_top_traffic_geographies(target_id: str, domain: str, top_n: int) -> List[str]:
                                                                                       ^^^^
NameError: name 'List' is not defined. Did you mean: 'list'?

